# Рисунки для статьи: интерактивное редактирование

Восемь многопанельных рисунков из прежнего `article_figures.py`:
три режима баланса и пять заданных углов. Код оформления панелей A–D
находится ниже, в отдельных ячейках.

1. Выберите ядро **SpikingNN (.venv)** и выполните **Run All**.
   По умолчанию строится рисунок A1 из готовой серии от 19 октября.
2. Изменяйте настройки, код панелей или готовые объекты `fig` и `axes`.
   После изменения функции панели повторите ячейку построения рисунка.
3. Панель управления позволяет выбрать сценарий, окно сигналов и толщину линий.
   При установленном `ipympl` доступны масштабирование и перемещение мышью.
4. Для всех восьми рисунков включите `BUILD_ALL`, для записи файлов — `EXPORT`.
   Экспорт сохраняет текущие объекты, включая ручные правки.

Тетрадка читает проверенные траектории; сигналы C/D восстанавливаются из
сохранённых состояний через исходное ядро без интегрирования новых эпизодов.
Полные данные имеют шаг **0.1 мс**; прореживание A/B влияет только на отображение.
Падения и последние отсчёты сохраняются. [Описание протокола](../ARTICLE_FIGURES.md).

In [ ]:
from pathlib import Path
from datetime import datetime
from copy import deepcopy
import importlib.util
import json
import sys

def locate_report():
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "report",
                          parent / "scripts/AfferentBalance2026/report"):
            if (candidate / "article_figure_data.py").is_file():
                return candidate.resolve()
    raise FileNotFoundError("Запустите Jupyter внутри репозитория SpikingNN.")

REPORT = locate_report()
sys.path.insert(0, str(REPORT))
from article_figure_data import figure_data, read_json, sha256, write_json

# Можно указать "inline" вручную для статичного вывода.
BACKEND = "widget" if importlib.util.find_spec("ipympl") else "inline"
get_ipython().run_line_magic("matplotlib", BACKEND)
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import Normalize
from matplotlib.patches import Rectangle
from matplotlib.backends.backend_pdf import PdfPages
from IPython.display import display, clear_output
import ipywidgets as widgets

FIGURES, AXES, FIGURE_SOURCES = {}, {}, {}
print("Каталог:", REPORT, "| Matplotlib:", BACKEND)
if BACKEND == "inline":
    print("Панель управления и редактирование в ячейках доступны. "
          "Для масштаба мышью установите requirements-notebook.txt и перезапустите ядро.")

## Данные и общие настройки

`SCENARIO` выбирает один рисунок. `SIGNAL_WINDOW` задаёт общее окно C/D,
`ANGLE_WINDOW` — диапазон времени B, `ZOOM_WINDOW` — его вставку.
Меняйте подписи прямо в функциях панелей. После изменения `STUDY`
выполните ячейки ниже заново; для другого численного ядра перезапустите kernel.

`STYLE` управляет шрифтом, `PHASE_STYLE` — линиями фазовых траекторий.
Times New Roman берётся из системных шрифтов или Wine; при его отсутствии
для просмотра используется DejaVu Serif.

In [ ]:
STUDY = REPORT / "results/article_phase_grid_20261019"
SCENARIO = "balance_A1"
FIGSIZE = (10.8, 9.2)
DISPLAY_STRIDE = 10          # A/B: 1 мс; начальная и последняя точки сохраняются.
SIGNAL_WINDOW = (0., 2.)     # C/D: все отсчёты по 0.1 мс в этом окне.
ANGLE_WINDOW = (0., 10.)
ZOOM_WINDOW = (0., .7)
PHASE_XLIM = (-92., 92.)
COLOR_MAP = "coolwarm"
MUSCLE_COLORS = ("#0072B2", "#D55E00")
PHASE_STYLE = dict(linewidth=.65, alpha=.60)
FONT = "Times New Roman"
for path in (Path.home() / ".wine/drive_c/windows/Fonts").glob("times*.ttf"):
    font_manager.fontManager.addfont(str(path))
try:
    font_manager.findfont(FONT, fallback_to_default=False)
except ValueError:
    FONT = "DejaVu Serif"
    print("Times New Roman не найден; используется", FONT)
STYLE = {
    "font.family": FONT, "font.size": 12,
    "axes.labelsize": 14, "axes.titlesize": 14,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 10,
    "mathtext.fontset": "stix", "pdf.fonttype": 42,
    "ps.fonttype": 42, "svg.fonttype": "none",
    "axes.spines.top": False, "axes.spines.right": False,
    "savefig.facecolor": "white",
}
plt.rcParams.update(STYLE)
PROTOCOL = read_json(STUDY / "protocol.json")
SCENARIOS = {spec["name"]: spec for spec in PROTOCOL["scenarios"]}
figure_data.cache_clear()
for name, spec in SCENARIOS.items():
    print(f"{name}: {spec['title']}")

## Сетка панелей

Все оси именованы: `A`, `B`, `zoom`, `C`, `force`, `D` (список из трёх осей).
Редактируйте размеры рисунка в `FIGSIZE`, пропорции и расстояния — здесь.

In [ ]:
def make_axes():
    fig = plt.figure(figsize=FIGSIZE, layout="constrained")
    grid = fig.add_gridspec(2, 2, height_ratios=(1.1, 1), hspace=.10, wspace=.08)
    axes = {"A": fig.add_subplot(grid[0, 0]), "B": fig.add_subplot(grid[0, 1])}
    cgrid = grid[1, 0].subgridspec(2, 1, hspace=.10)
    axes["C"] = fig.add_subplot(cgrid[0, 0])
    axes["force"] = fig.add_subplot(cgrid[1, 0], sharex=axes["C"])
    dgrid = grid[1, 1].subgridspec(3, 1, hspace=.06)
    axes["D"] = [fig.add_subplot(dgrid[i, 0]) for i in range(3)]
    return fig, axes

## A — фазовые траектории

100 стартов; цвет кодирует начальный угол. Крест обозначает досрочное завершение,
звезда — цель. Здесь можно менять линии, маркеры, подписи и границы осей.

In [ ]:
def draw_phase(ax, data, cmap, norm):
    target = data["spec"]["target_deg"]
    ax.add_patch(Rectangle((target - 3., -.35), 6., .7,
                           color="#77AA55", alpha=.16, zorder=0))
    for row, trace in zip(data["phase"], data["phase_lines"]):
        color = cmap(norm(row["theta0_deg"]))
        ax.plot(np.rad2deg(trace[:, 1]), trace[:, 2], color=color,
                rasterized=False, **PHASE_STYLE)
        ax.scatter(row["theta0_deg"], row["omega0_rad_s"], s=9, c=[color],
                   edgecolors="#444444", linewidths=.22, zorder=4)
        if row["termination"] != "duration":
            ax.plot(np.rad2deg(trace[-1, 1]), trace[-1, 2], "x",
                    color="#333333", ms=3, mew=.6)
    ax.scatter(target, 0., marker="*", color="black", s=110, zorder=6)
    successes = sum(row["success"] for row in data["phase"])
    ax.set(xlabel=r"Angle from upright $\theta$ (deg)",
           ylabel=r"Angular velocity $\omega$ (rad/s)",
           title=f"A   Phase trajectories: {successes}/100 successful",
           xlim=PHASE_XLIM)

## B — угол во времени и увеличенная вставка

Десять стартов с нулевой скоростью. Толстая линия соответствует заранее
выбранному примеру C/D; оформление не меняет выбор траектории.

In [ ]:
def draw_angles(ax, data, cmap, norm):
    target = data["spec"]["target_deg"]
    inset_y = .05 if target >= 0. else .57
    zoom = ax.inset_axes([.40, inset_y, .57, .35])
    zoom.set_facecolor("#FFFFFF")
    for row, trace in zip(data["zero"], data["zero_lines"]):
        highlighted = row["trial"] == data["example_record"]["trial"]
        color = cmap(norm(row["theta0_deg"]))
        ax.plot(trace[:, 0], np.rad2deg(trace[:, 1]), color=color,
                lw=2.3 if highlighted else 1.05, alpha=1 if highlighted else .8)
        early = (trace[:, 0] >= ZOOM_WINDOW[0]) & (trace[:, 0] <= ZOOM_WINDOW[1])
        zoom.plot(trace[early, 0], np.rad2deg(trace[early, 1]),
                  color=color, lw=1.8 if highlighted else .8)
        if row["termination"] != "duration":
            ax.plot(trace[-1, 0], np.rad2deg(trace[-1, 1]), "x", c="#333333", ms=5)
            if ZOOM_WINDOW[0] <= trace[-1, 0] <= ZOOM_WINDOW[1]:
                zoom.plot(trace[-1, 0], np.rad2deg(trace[-1, 1]), "x", c="#333333", ms=3)
    ax.axhline(target, color="black", ls="--", lw=1)
    ax.axhspan(target - 3., target + 3., color="#77AA55", alpha=.13)
    successes = sum(row["success"] for row in data["zero"])
    ax.set(xlabel="Time (s)", ylabel=r"Angle $\theta$ (deg)", xlim=ANGLE_WINDOW,
           title=rf"B   Ten starts at $\omega_0=0$: {successes}/10 successful")
    zoom.axhline(target, color="black", ls="--", lw=.7)
    zoom.set(xlim=ZOOM_WINDOW, title=f"{ZOOM_WINDOW[0]:g}–{ZOOM_WINDOW[1]:g} s")
    zoom.title.set_fontsize(10)
    zoom.tick_params(labelsize=9)
    zoom.grid(alpha=.15)
    return zoom

## C/D — активация, силы и афференты

Один и тот же старт во всех рисунках: θ₀ ≈ −27.78°, ω₀ = 0.
Восстановленные нелинейные слагаемые проверяются по сохранённым афферентам.
При падении линии заканчиваются вместе с траекторией.

In [ ]:
def draw_signals(axes, data):
    trace = data["example"]["trace"]
    window = (trace[:, 0] >= SIGNAL_WINDOW[0]) & (trace[:, 0] <= SIGNAL_WINDOW[1])
    ax_c, ax_force, d_axes = axes["C"], axes["force"], axes["D"]
    for i, (label, color) in enumerate(zip(("Flexor", "Extensor"), MUSCLE_COLORS)):
        ax_c.plot(trace[window, 0], data["signals"]["activation"][window, i],
                  c=color, lw=1.1, label=label)
        ax_force.plot(trace[window, 0], trace[window, 3 + i], c=color, lw=1.1)
        for j, ax in enumerate(d_axes):
            ax.plot(trace[window, 0], trace[window, 11 + 3 * i + j], c=color, lw=1.0)
    ax_c.set(title="C   One example: muscle activation and force",
             ylabel=r"Activation $x$", ylim=(-.03, 1.03))
    ax_c.legend(loc="upper right", ncol=2, frameon=False,
                handlelength=1.2, columnspacing=.8)
    ax_force.set(xlabel="Time (s)", ylabel="Force (N)")
    ax_c.tick_params(labelbottom=False)
    d_axes[0].set_title("D   Same example: afferents (model units)")
    # Если окно целиком после падения, оставляем пустые оси с выбранным диапазоном.
    end = min(SIGNAL_WINDOW[1], float(trace[-1, 0]))
    limits = (SIGNAL_WINDOW[0], end) if end > SIGNAL_WINDOW[0] else SIGNAL_WINDOW
    for ax, channel in zip(d_axes, ("Ia", "II", "Ib")):
        ax.set_ylabel(channel)
        ax.set_xlim(limits)
        if ax is not d_axes[-1]:
            ax.tick_params(labelbottom=False)
    d_axes[-1].set_xlabel("Time (s)")
    ax_force.set_xlim(limits)

## Сборка рисунка и общие подписи

Эта функция возвращает редактируемые `Figure` и оси. Повторное построение
заменяет объект соответствующего сценария в `FIGURES`; остальные остаются доступны.

In [ ]:
def build_figure(name):
    data = figure_data(STUDY.resolve(), name, DISPLAY_STRIDE)
    cmap, norm = plt.get_cmap(COLOR_MAP), Normalize(-50., 50.)
    with plt.ioff(), plt.rc_context(STYLE):
        fig, axes = make_axes()
        draw_phase(axes["A"], data, cmap, norm)
        axes["zoom"] = draw_angles(axes["B"], data, cmap, norm)
        draw_signals(axes, data)
        for ax in [axes["A"], axes["B"], axes["C"], axes["force"], *axes["D"]]:
            ax.grid(alpha=.18, lw=.6)
        colorbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                               ax=[axes["A"], axes["B"]], location="bottom",
                               shrink=.84, aspect=45, pad=.015)
        colorbar.set_label(r"Initial angle $\theta_0$ (deg); same colors in A and B",
                          fontsize=11)
        colorbar.set_ticks(np.linspace(-50, 50, 5))
        axes["colorbar"] = colorbar.ax
        example = data["example_record"]
        outcome = "success" if example["success"] else "not successful"
        fig.suptitle(data["spec"]["title"] + "\n" +
                     r"$\Delta t=0.1$ ms; torque factor 1; no PD; " +
                     rf"C/D: $\theta_0={example['theta0_deg']:.2f}^\circ$, $\omega_0=0$ " +
                     f"({outcome})", fontsize=16)
    if name in FIGURES:
        plt.close(FIGURES[name])
    FIGURES[name], AXES[name] = fig, axes
    FIGURE_SOURCES[name] = {
        "study": str(STUDY.resolve()),
        "protocol_sha256": sha256(STUDY / "protocol.json"),
        "traces_manifest_sha256": sha256(STUDY / "traces.json"),
        "example_file": example["file"], "example_sha256": example["sha256"],
        "example_outcome": outcome, "nonlinear_terms_verified": True,
        "display_stride": DISPLAY_STRIDE, "signal_display_step_ms": .1,
        "signal_window": list(SIGNAL_WINDOW), "angle_window": list(ANGLE_WINDOW),
        "zoom_window": list(ZOOM_WINDOW), "figsize": list(FIGSIZE),
        "style_at_build": deepcopy(STYLE), "phase_style": deepcopy(PHASE_STYLE),
        "color_map": COLOR_MAP, "muscle_colors": list(MUSCLE_COLORS),
    }
    return fig, axes

def show_figure(fig):
    display(fig.canvas if BACKEND == "widget" else fig)
    if BACKEND == "inline":
        plt.close(fig)  # Объект остаётся в FIGURES и доступен для правки/экспорта.

## Построить выбранный рисунок

Повторите эту ячейку после изменения настроек или функций выше.
Повторное оформление использует кэш чтения данных; интегрирование не запускается.

In [ ]:
fig, axes = build_figure(SCENARIO)
show_figure(fig)

## Интерактивная панель управления

Выберите параметры и нажмите **Перерисовать**. Кнопка обновляет `SCENARIO`,
`SIGNAL_WINDOW`, `PHASE_STYLE`, а также `fig` и `axes`. Для правки уже
построенного рисунка без его пересоздания используйте следующую ячейку.

In [ ]:
scenario_control = widgets.Dropdown(
    options=[(spec["title"], name) for name, spec in SCENARIOS.items()],
    value=SCENARIO, description="Сценарий:", layout=widgets.Layout(width="650px"))
window_control = widgets.FloatRangeSlider(
    value=SIGNAL_WINDOW, min=0., max=float(PROTOCOL["duration_s"]), step=.1,
    description="C/D, с:", continuous_update=False, layout=widgets.Layout(width="650px"))
width_control = widgets.FloatSlider(
    value=PHASE_STYLE["linewidth"], min=.1, max=3., step=.05,
    description="Линии A:", continuous_update=False)
redraw_button = widgets.Button(description="Перерисовать", button_style="primary")
preview = widgets.Output()

def redraw(_):
    global SCENARIO, SIGNAL_WINDOW, fig, axes
    with preview:
        clear_output(wait=True)
        if window_control.value[0] >= window_control.value[1]:
            print("Задайте окно C/D ненулевой ширины.")
            return
        SCENARIO = scenario_control.value
        SIGNAL_WINDOW = tuple(window_control.value)
        PHASE_STYLE["linewidth"] = width_control.value
        fig, axes = build_figure(SCENARIO)
        show_figure(fig)

redraw_button.on_click(redraw)
display(widgets.VBox([scenario_control, window_control, width_control, redraw_button, preview]))

## Ручная правка готового рисунка

Раскомментируйте нужные строки. В режиме `widget` правки видны сразу после
`draw_idle()`; в `inline` вызовите `show_figure(fig)`.
Можно выбрать ранее построенный рисунок через `FIGURES[name]` и `AXES[name]`.
Повторное построение из функций отменяет ручные правки этого рисунка.

In [ ]:
# fig, axes = FIGURES[SCENARIO], AXES[SCENARIO]
# fig.set_size_inches(12, 9)
# fig.suptitle("Баланс: большие начальные углы", fontsize=18)
# axes["A"].set(xlim=(-60, 60), xlabel="Угол, °", ylabel="Угловая скорость, рад/с")
# axes["B"].set_title("Угол во времени")
# axes["zoom"].set_xlim(0, .4)
# axes["C"].lines[0].set_color("crimson")
# axes["D"][0].set_ylabel("Ia")
# fig.canvas.draw_idle()
# if BACKEND == "inline":
#     show_figure(fig)

## Построить все восемь рисунков

Включите `BUILD_ALL` для подготовки всей серии. Уже построенные рисунки
сохраняются с ручными правками. Для их пересоздания включите также `REBUILD_EXISTING`.

In [ ]:
BUILD_ALL = False
REBUILD_EXISTING = False
if BUILD_ALL:
    for name in SCENARIOS:
        if name not in FIGURES or REBUILD_EXISTING:
            built, _ = build_figure(name)
            show_figure(built)
            print("Построен:", name)
print("Доступны:", ", ".join(FIGURES))

## Экспорт текущих рисунков

Включите `EXPORT`, выберите `EXPORT_NAMES` и повторите ячейку. Файлы создаются
в новом каталоге `report/figures_edited/article_<дата-время>/`.
PDF/SVG сохраняют векторные линии, PNG — растровый просмотр.
`all_panels.pdf` содержит по странице на выбранный рисунок.
Исходная серия и её иллюстрации не перезаписываются.

Перед экспортом сохраните тетрадку: её текущая версия на диске копируется
рядом с рисунками. `export.json` записывает источники, настройки построения
и контрольные суммы; последующие ручные правки объектов сохраняются в изображениях.

In [ ]:
def export_figures(names, formats=("pdf", "png", "svg"), dpi=240, root=None):
    names, formats = list(dict.fromkeys(names)), tuple(formats)
    if not names or any(name not in FIGURES for name in names):
        raise ValueError("Выберите уже построенные рисунки из FIGURES.")
    if not formats or any(fmt not in ("pdf", "png", "svg") for fmt in formats):
        raise ValueError("Поддерживаются pdf, png и svg.")
    root = Path(root) if root is not None else REPORT / "figures_edited"
    destination = root / ("article_" + datetime.now().strftime("%Y%m%d_%H%M%S_%f"))
    destination.mkdir(parents=True, exist_ok=False)
    for name in names:
        for suffix in formats:
            FIGURES[name].savefig(destination / f"{name}.{suffix}", dpi=dpi)
    if "pdf" in formats:
        with PdfPages(destination / "all_panels.pdf") as bundle:
            for name in names:
                bundle.savefig(FIGURES[name])
    notebook = REPORT / "article_figures.ipynb"
    (destination / notebook.name).write_bytes(notebook.read_bytes())
    metadata = {
        "figures": names, "formats": formats, "dpi": dpi,
        "sources": {name: FIGURE_SOURCES[name] for name in names},
        "files": {path.name: sha256(path) for path in sorted(destination.iterdir())},
        "note": "Current Figure objects, including manual edits; no new simulations.",
    }
    write_json(destination / "export.json", metadata)
    return destination

EXPORT = False
EXPORT_NAMES = list(FIGURES)  # Например: ["balance_A1", "setpoint_plus_22p5"].
EXPORT_FORMATS = ("pdf", "png", "svg")
if EXPORT:
    print("Сохранено:", export_figures(EXPORT_NAMES, EXPORT_FORMATS))
else:
    print("Экспорт выключен. Для записи текущих рисунков включите EXPORT.")

## Пересчёт исходных траекторий (отдельный запуск)

Вспомогательный модуль `article_figure_data.py` сохраняет расчёт сетки
и независимую библиотечную проверку. Запускайте из корня репозитория:

```bash
OPENBLAS_NUM_THREADS=1 OMP_NUM_THREADS=1 .venv/bin/python \
  scripts/AfferentBalance2026/report/article_figure_data.py all \
  --output scripts/AfferentBalance2026/report/results/article_phase_grid_new \
  --workers 6
```

`compute` рассчитывает 880 эпизодов, `verify` проверяет сохранённые примеры,
`all` выполняет оба действия. После расчёта укажите новый каталог в `STUDY`.
Обычный **Run All** тетрадки эти команды не вызывает.